# YBT Data Processing Pipeline

This notebook processes the YBT dataset using the exact same methodology as the `data_pipeline_recreation.ipynb` notebook, adapted for YBT-specific characteristics.

## YBT-Specific Adaptations
- **No SPQ data**: The YBT dataset does not contain SPQ questionnaire data
- **Target variable**: Autism target identified by selection of 'autism' in diagnosis column
- **Available questionnaires**: EQ-10, SQR-10, AQ-10 (no SPQ-10)

## Pipeline Overview
1. **Initial Data Exploration**: Understand YBT dataset structure and characteristics
2. **Raw data loading and initial processing**
3. **Target variable creation**: YBT-specific autism diagnosis logic
4. **Missing value handling** (with corrected sex imputation)
5. **Questionnaire scoring** (EQ, SQR, AQ - no SPQ)
6. **Feature engineering** (excluding SPQ-related features)
7. **Data standardization and encoding**
8. **Data balancing** (50/50 split)
9. **Final dataset filtering** (exclude autism cases with AQ < 6)
10. **Experimental setups** adapted for YBT

## Expected Results
- Scientifically rigorous processing pipeline
- Publication-ready validation framework
- Comprehensive model performance analysis
- Clinical interpretation of YBT-specific findings


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score, 
    f1_score, precision_score, recall_score, precision_recall_curve
)
from sklearn.utils import resample

# Advanced ML libraries
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 0. INITIAL YBT DATA EXPLORATION

Before implementing the processing pipeline, we need to understand the YBT dataset structure and characteristics.


In [ ]:
print("="*80)
print("STEP 0: YBT DATASET INITIAL EXPLORATION")
print("="*80)

# Load YBT data
ybt_data_path = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv'
print(f"Loading YBT data from: {ybt_data_path}")

try:
    df_ybt = pd.read_csv(ybt_data_path)
    print(f"YBT dataset shape: {df_ybt.shape}")
    print(f"YBT columns: {list(df_ybt.columns)}")
    
    # Basic dataset information
    print(f"\nDataset Info:")
    print(f"  Total rows: {df_ybt.shape[0]:,}")
    print(f"  Total columns: {df_ybt.shape[1]}")
    print(f"  Memory usage: {df_ybt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for questionnaire columns (YBT-specific naming)
    spq_cols = [col for col in df_ybt.columns if col.startswith('spq_')]
    eq_cols = [col for col in df_ybt.columns if col.startswith('eq10_')]
    sqr_cols = [col for col in df_ybt.columns if col.startswith('sq10_')]
    aq_cols = [col for col in df_ybt.columns if col.startswith('aq_')]
    
    print(f"\nQuestionnaire columns found:")
    print(f"  SPQ columns: {len(spq_cols)} - {spq_cols[:5] if spq_cols else 'None'}")
    print(f"  EQ columns: {len(eq_cols)} - {eq_cols[:5] if eq_cols else 'None'}")
    print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols[:5] if sqr_cols else 'None'}")
    print(f"  AQ columns: {len(aq_cols)} - {aq_cols[:5] if aq_cols else 'None'}")
    
    # Look for diagnosis/target columns
    diagnosis_cols = [col for col in df_ybt.columns if 'diagnosis' in col.lower() or 'autism' in col.lower()]
    print(f"\nDiagnosis/autism columns: {diagnosis_cols}")
    
    # Check for demographic columns (YBT-specific naming)
    demo_cols = ['age', 'sex', 'gender', 'hand', 'edu', 'country']
    available_demo = [col for col in demo_cols if col in df_ybt.columns]
    print(f"Available demographic columns: {available_demo}")
    
    # CRITICAL: Show actual row examples to understand data structure
    print(f"\n" + "="*80)
    print("ACTUAL ROW EXAMPLES - UNDERSTANDING DATA STRUCTURE")
    print("="*80)
    
    # Show first few rows of key columns
    key_cols = ['age', 'sex', 'gender', 'diagnosis_yes_no', 'diagnosis']
    print(f"\nFirst 5 rows of key columns:")
    for col in key_cols:
        if col in df_ybt.columns:
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # Show questionnaire data examples
    if eq_cols:
        print(f"\nFirst 5 rows of EQ questionnaire data:")
        for col in eq_cols[:3]:  # Show first 3 EQ columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    if sqr_cols:
        print(f"\nFirst 5 rows of SQR questionnaire data:")
        for col in sqr_cols[:3]:  # Show first 3 SQR columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    if aq_cols:
        print(f"\nFirst 5 rows of AQ questionnaire data:")
        for col in aq_cols[:3]:  # Show first 3 AQ columns
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # Show diagnosis data examples
    if diagnosis_cols:
        print(f"\nFirst 5 rows of diagnosis data:")
        for col in diagnosis_cols:
            sample_vals = df_ybt[col].head(5).tolist()
            print(f"  {col}: {sample_vals}")
    
    # CRITICAL: Analyze data types and values
    print(f"\n" + "="*80)
    print("DATA TYPE AND VALUE ANALYSIS")
    print("="*80)
    
    # Check data types
    print(f"\nData types:")
    dtypes = df_ybt.dtypes.value_counts()
    print(dtypes)
    
    # Check for text vs numeric data in questionnaire columns
    all_questionnaire_cols = eq_cols + sqr_cols + aq_cols
    if all_questionnaire_cols:
        print(f"\nQuestionnaire data analysis:")
        sample_col = all_questionnaire_cols[0]
        print(f"Sample column: {sample_col}")
        
        # Show unique values
        unique_vals = df_ybt[sample_col].value_counts(dropna=False).head(10)
        print(f"Unique values in {sample_col}: {unique_vals.to_dict()}")
        
        # Check if it's text responses
        sample_values = df_ybt[sample_col].dropna().head(10).tolist()
        are_text = any(isinstance(val, str) and any(word in val.lower() for word in ['agree', 'disagree', 'strongly', 'slightly']) for val in sample_values)
        print(f"Contains text responses (agree/disagree): {are_text}")
        
        # Check if it's already numeric
        are_numeric = all(pd.api.types.is_numeric_dtype(df_ybt[col]) for col in all_questionnaire_cols[:3])
        print(f"All questionnaire columns are numeric: {are_numeric}")
        
        # Check if it's binary (0-1)
        if are_numeric:
            sample_binary = df_ybt[sample_col].dropna().head(10).tolist()
            are_binary = all(val in [0, 1] for val in sample_binary if pd.notna(val))
            print(f"Is binary (0-1): {are_binary}")
            
            if are_binary:
                print("✅ Data is already binary - no scoring needed")
            else:
                print("✅ Data is numeric but not binary - scoring needed")
        else:
            print("✅ Data contains text - conversion needed")
    
    # Check missing data patterns
    print(f"\nMissing data summary:")
    missing_data = df_ybt.isnull().sum().sort_values(ascending=False)
    print(f"Columns with missing data: {len(missing_data[missing_data > 0])}")
    if len(missing_data[missing_data > 0]) > 0:
        print("Top 10 columns with most missing data:")
        print(missing_data.head(10))
    
    # Check for metadata contamination
    print(f"\nMetadata contamination check:")
    metadata_cols = []
    for col in df_ybt.columns:
        if df_ybt[col].astype(str).str.contains('ImportId|question', case=False, na=False).any():
            metadata_cols.append(col)
    
    if metadata_cols:
        print(f"Columns with metadata contamination: {len(metadata_cols)}")
        print(f"Sample contaminated columns: {metadata_cols[:5]}")
        
        # Show example metadata
        for col in metadata_cols[:2]:
            sample_metadata = df_ybt[col].astype(str).str.contains('ImportId|question', case=False, na=False)
            metadata_examples = df_ybt[sample_metadata][col].head(3).tolist()
            print(f"  {col} metadata examples: {metadata_examples}")
    else:
        print("No metadata contamination detected")
    
    print(f"\n" + "="*80)
    print("EXPLORATION COMPLETE - READY FOR PROCESSING")
    print("="*80)
    
except FileNotFoundError:
    print(f"ERROR: YBT data file not found at {ybt_data_path}")
    print("Please check the file path and ensure the YBT.csv file exists.")
    print("Creating dummy dataset for demonstration...")
    
    # Create dummy dataset for demonstration
    np.random.seed(42)
    n_samples = 1000
    
    df_ybt = pd.DataFrame({
        'age': np.random.randint(18, 80, n_samples),
        'sex': np.random.choice([1, 2], n_samples),
        'eq_1': np.random.randint(1, 5, n_samples),
        'eq_2': np.random.randint(1, 5, n_samples),
        'eq_3': np.random.randint(1, 5, n_samples),
        'eq_4': np.random.randint(1, 5, n_samples),
        'eq_5': np.random.randint(1, 5, n_samples),
        'eq_6': np.random.randint(1, 5, n_samples),
        'eq_7': np.random.randint(1, 5, n_samples),
        'eq_8': np.random.randint(1, 5, n_samples),
        'eq_9': np.random.randint(1, 5, n_samples),
        'eq_10': np.random.randint(1, 5, n_samples),
        'sqr_1': np.random.randint(1, 5, n_samples),
        'sqr_2': np.random.randint(1, 5, n_samples),
        'sqr_3': np.random.randint(1, 5, n_samples),
        'sqr_4': np.random.randint(1, 5, n_samples),
        'sqr_5': np.random.randint(1, 5, n_samples),
        'sqr_6': np.random.randint(1, 5, n_samples),
        'sqr_7': np.random.randint(1, 5, n_samples),
        'sqr_8': np.random.randint(1, 5, n_samples),
        'sqr_9': np.random.randint(1, 5, n_samples),
        'sqr_10': np.random.randint(1, 5, n_samples),
        'aq_1': np.random.randint(1, 5, n_samples),
        'aq_2': np.random.randint(1, 5, n_samples),
        'aq_3': np.random.randint(1, 5, n_samples),
        'aq_4': np.random.randint(1, 5, n_samples),
        'aq_5': np.random.randint(1, 5, n_samples),
        'aq_6': np.random.randint(1, 5, n_samples),
        'aq_7': np.random.randint(1, 5, n_samples),
        'aq_8': np.random.randint(1, 5, n_samples),
        'aq_9': np.random.randint(1, 5, n_samples),
        'aq_10': np.random.randint(1, 5, n_samples),
    })
    
    # Add diagnosis column with autism selection
    diagnosis_options = ['autism', 'adhd', 'anxiety', 'depression', 'none']
    df_ybt['diagnosis_selection'] = df_ybt.apply(
        lambda x: 'autism' if np.random.random() < 0.1 else np.random.choice(diagnosis_options), 
        axis=1
    )
    
    print(f"Dummy YBT dataset created with shape: {df_ybt.shape}")
    print(f"Dummy columns: {list(df_ybt.columns)}")

## 1. YBT DATA PROCESSING PIPELINE

### A. Raw Data Loading and Initial Processing


In [ ]:
print("="*80)
print("STEP A: YBT RAW DATA LOADING AND INITIAL PROCESSING")
print("="*80)

# Use the YBT dataset from exploration
df = df_ybt.copy()
print(f"Starting with YBT dataset shape: {df.shape}")

# CRITICAL: Drop metadata rows (first 2 rows) from ALL columns
print("\nDropping metadata rows (first 2 rows) from all columns...")
df = df.iloc[2:].reset_index(drop=True)
print(f"After dropping metadata rows: {df.shape}")

# Check what the clean data looks like now
print(f"\nSample of clean data:")
sample_cols = ['age', 'sex', 'diagnosis_yes_no', 'diagnosis', 'eq10_1', 'aq_1']
for col in sample_cols:
    if col in df.columns:
        sample_vals = df[col].dropna().head(3).tolist()
        print(f"  {col}: {sample_vals}")

# Remove columns with mostly missing data
print(f"\nAnalyzing missing data patterns...")
missing_data = df.isnull().sum().sort_values(ascending=False)
total_rows = len(df)

print(f"Missing data analysis:")
print(f"  Total rows: {total_rows}")
print(f"  Columns with missing data: {len(missing_data[missing_data > 0])}")

# Define threshold for dropping columns (e.g., >80% missing)
missing_threshold = 0.80  # Drop columns with >80% missing data
print(f"  Missing data threshold: {missing_threshold*100}%")

# Identify columns to drop
columns_to_drop = []
for col in df.columns:
    missing_count = missing_data[col]
    missing_percentage = missing_count / total_rows
    
    if missing_percentage > missing_threshold:
        columns_to_drop.append(col)
        print(f"    DROP: {col} - {missing_count}/{total_rows} ({missing_percentage*100:.1f}% missing)")

print(f"\nDropping {len(columns_to_drop)} columns with >{missing_threshold*100}% missing data...")
df = df.drop(columns=columns_to_drop)
print(f"After dropping high missing columns: {df.shape}")

# Check remaining missing data
print(f"\nRemaining missing data summary:")
remaining_missing = df.isnull().sum().sort_values(ascending=False)
print(f"  Columns with missing data: {len(remaining_missing[remaining_missing > 0])}")
if len(remaining_missing[remaining_missing > 0]) > 0:
    print("  Top 10 columns with most missing data:")
    print(remaining_missing.head(10))

# Check data types after cleaning
print(f"\nData types after cleaning:")
dtypes = df.dtypes.value_counts()
print(dtypes)

# Check questionnaire columns availability
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"\nQuestionnaire columns after cleaning:")
print(f"  EQ columns: {len(eq_cols)} - {eq_cols}")
print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols}")
print(f"  AQ columns: {len(aq_cols)} - {aq_cols}")

# Check diagnosis columns
diagnosis_cols = [col for col in df.columns if 'diagnosis' in col.lower()]
print(f"  Diagnosis columns: {diagnosis_cols}")

# Show sample values from clean questionnaire data
if eq_cols:
    print(f"\nSample clean questionnaire values:")
    sample_col = eq_cols[0]
    sample_vals = df[sample_col].dropna().head(5).tolist()
    print(f"  {sample_col}: {sample_vals}")

# Show sample values from clean diagnosis data
if diagnosis_cols:
    print(f"\nSample clean diagnosis values:")
    for col in diagnosis_cols:
        sample_vals = df[col].dropna().head(5).tolist()
        print(f"  {col}: {sample_vals}")

# ADDED: Print all column names
print(f"\n" + "="*80)
print("ALL COLUMN NAMES AFTER CLEANING")
print("="*80)
print(f"Total columns: {len(df.columns)}")
print(f"Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nStep A complete. Dataset shape: {df.shape}")
print("✅ Metadata rows removed")
print("✅ High missing data columns removed")
print("✅ Data ready for target variable creation")

### B. Missing Value Handling


In [ ]:
print("="*80)
print("STEP B: YBT MISSING VALUE HANDLING")
print("="*80)

# Clean age column first
if 'age' in df.columns:
    print("Cleaning age column...")
    # Convert to numeric
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    print(f"Age column cleaned. Sample values: {df['age'].dropna().head(5).tolist()}")

# Clean demographic columns
demographic_cols = ['sex', 'gender', 'hand', 'edu', 'country']
available_demo_cols = [col for col in demographic_cols if col in df.columns]
print(f"\nCleaning demographic columns: {available_demo_cols}")

for col in available_demo_cols:
    if col in df.columns:
        # Fill missing values with 'unknown'
        df[col] = df[col].fillna('unknown')
        print(f"  {col}: {df[col].isnull().sum()} missing values remaining")

# Focus on questionnaire missing data analysis
questionnaire_cols = [col for col in df.columns if any(q in col for q in ['eq10_', 'sq10_', 'aq_'])]
print(f"\nQUESTIONNAIRE MISSING DATA ANALYSIS: {len(questionnaire_cols)} columns")

if questionnaire_cols:
    # Analyze missing data patterns in questionnaire responses
    print(f"\nMissing data analysis by questionnaire type:")
    
    # Calculate missing data for each questionnaire type
    for q_type in ['eq10_', 'sq10_', 'aq_']:
        q_cols = [col for col in questionnaire_cols if col.startswith(q_type)]
        if q_cols:
            # Calculate missing data for this questionnaire type
            q_missing = df[q_cols].isnull().sum(axis=1)
            q_total = len(q_cols)
            q_missing_percentage = (q_missing / q_total) * 100
            
            print(f"\n{q_type} questionnaire analysis:")
            print(f"  Total questions: {q_total}")
            print(f"  Mean missing per person: {q_missing_percentage.mean():.1f}%")
            print(f"  Median missing per person: {q_missing_percentage.median():.1f}%")
            print(f"  Max missing per person: {q_missing_percentage.max():.1f}%")
            
            # Show distribution of missing data
            missing_distribution = q_missing_percentage.value_counts().sort_index()
            print(f"  Missing data distribution:")
            print(f"    0% missing: {missing_distribution.get(0.0, 0)} people")
            print(f"    1-25% missing: {q_missing_percentage[(q_missing_percentage > 0) & (q_missing_percentage <= 25)].count()} people")
            print(f"    26-50% missing: {q_missing_percentage[(q_missing_percentage > 25) & (q_missing_percentage <= 50)].count()} people")
            print(f"    51-75% missing: {q_missing_percentage[(q_missing_percentage > 50) & (q_missing_percentage <= 75)].count()} people")
            print(f"    76-100% missing: {q_missing_percentage[q_missing_percentage > 75].count()} people")
    
    # Calculate overall missing data per person across ALL questionnaires
    print(f"\nOVERALL QUESTIONNAIRE MISSING DATA ANALYSIS:")
    all_q_missing = df[questionnaire_cols].isnull().sum(axis=1)
    all_q_total = len(questionnaire_cols)
    all_q_missing_percentage = (all_q_missing / all_q_total) * 100
    
    print(f"  Total questionnaire questions: {all_q_total}")
    print(f"  Mean missing per person: {all_q_missing_percentage.mean():.1f}%")
    print(f"  Median missing per person: {all_q_missing_percentage.median():.1f}%")
    print(f"  Max missing per person: {all_q_missing_percentage.max():.1f}%")
    
    # Show overall distribution
    overall_distribution = all_q_missing_percentage.value_counts().sort_index()
    print(f"  Overall missing data distribution:")
    print(f"    0% missing: {overall_distribution.get(0.0, 0)} people")
    print(f"    1-25% missing: {all_q_missing_percentage[(all_q_missing_percentage > 0) & (all_q_missing_percentage <= 25)].count()} people")
    print(f"    26-50% missing: {all_q_missing_percentage[(all_q_missing_percentage > 25) & (all_q_missing_percentage <= 50)].count()} people")
    print(f"    51-75% missing: {all_q_missing_percentage[(all_q_missing_percentage > 50) & (all_q_missing_percentage <= 75)].count()} people")
    print(f"    76-100% missing: {all_q_missing_percentage[all_q_missing_percentage > 75].count()} people")
    
    # DECISION: Drop individuals with high missing questionnaire data
    print(f"\nDECISION: Dropping individuals with high missing questionnaire data...")
    
    # Define threshold for dropping individuals
    # Best practice: Drop individuals with >50% missing questionnaire data
    # This ensures we keep people who answered most questions
    high_missing_threshold = 50  # Drop individuals with >50% missing questionnaire data
    
    print(f"  Threshold: Drop individuals with >{high_missing_threshold}% missing questionnaire data")
    print(f"  Rationale: Keep people who answered most questions for reliable analysis")
    
    # Identify individuals to drop
    high_missing_individuals = all_q_missing_percentage > high_missing_threshold
    individuals_to_drop = high_missing_individuals.sum()
    
    print(f"  Individuals with >{high_missing_threshold}% missing data: {individuals_to_drop}")
    print(f"  Individuals with <{high_missing_threshold}% missing data: {(all_q_missing_percentage <= high_missing_threshold).sum()}")
    
    if individuals_to_drop > 0:
        print(f"  Dropping {individuals_to_drop} individuals with high missing data...")
        df_before_drop = df.copy()
        df = df[~high_missing_individuals]
        rows_dropped = len(df_before_drop) - len(df)
        print(f"  Rows dropped: {rows_dropped}")
        print(f"  After dropping high missing data individuals: {df.shape}")
        
        # Check if we still have enough data
        if len(df) < 1000:
            print("  WARNING: Very few rows remaining after dropping high missing data individuals")
            print("  Consider using a lower threshold or imputation instead")
        else:
            print(f"  Sufficient data remaining: {len(df)} rows")
    else:
        print(f"  No individuals with >{high_missing_threshold}% missing data found")
        print(f"  Keeping all {len(df)} individuals")
        
else:
    print("No questionnaire columns found")

# Check remaining missing values
print(f"\nRemaining missing values after processing:")
remaining_missing = df.isnull().sum().sort_values(ascending=False)
print(f"  Columns with missing data: {len(remaining_missing[remaining_missing > 0])}")
if len(remaining_missing[remaining_missing > 0]) > 0:
    print("  Top 10 columns with most missing data:")
    print(remaining_missing.head(10))

print(f"\nStep B complete. Dataset shape: {df.shape}")
print("✅ High missing data individuals removed")
print("✅ Data ready for questionnaire scoring")

### C. Questionnaire Scoring and Totals (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("STEP C: YBT QUESTIONNAIRE SCORING")
print("="*80)

# Define response mapping for all questionnaires
print("Setting up response mapping...")

# Response mapping: text responses to numeric values
response_mapping = {
    'strongly agree': 4,
    'slightly agree': 3, 
    'slightly disagree': 2,
    'strongly disagree': 1
}

print(f"Response mapping: {response_mapping}")

# Get questionnaire columns
eq_cols = [col for col in df.columns if col.startswith('eq10_')]
sqr_cols = [col for col in df.columns if col.startswith('sq10_')]
aq_cols = [col for col in df.columns if col.startswith('aq_')]

print(f"\nQuestionnaire columns found:")
print(f"  EQ columns: {len(eq_cols)} - {eq_cols}")
print(f"  SQR columns: {len(sqr_cols)} - {sqr_cols}")
print(f"  AQ columns: {len(aq_cols)} - {aq_cols}")

# STEP 1: Convert text responses to numeric for all questionnaires
print(f"\nSTEP 1: Converting text responses to numeric...")

all_questionnaire_cols = eq_cols + sqr_cols + aq_cols
for col in all_questionnaire_cols:
    if col in df.columns:
        # Convert text responses to numeric using mapping
        df[col] = df[col].map(response_mapping)
        print(f"  Converted {col}: {df[col].dropna().head(3).tolist()}")

# STEP 2: EQ-10 Scoring (Binary 0-1 with reverse-scoring)
print(f"\nSTEP 2: EQ-10 Scoring (Binary 0-1)...")

if eq_cols:
    # EQ-10 scoring: Binary 0-1 per item
    # Items 1,2,4,5,6,7,8,9,10: Agree responses (3,4) = 1 point
    # Items 3: Disagree responses (1,2) = 1 point (reverse-scored)
    
    eq_reverse_items = [3]  # Items that need reverse scoring
    
    for i, col in enumerate(eq_cols, 1):
        if col in df.columns:
            if i in eq_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate EQ total
    df['eq_total'] = df[eq_cols].sum(axis=1)
    print(f"  EQ total range: {df['eq_total'].min()} to {df['eq_total'].max()}")
    print(f"  EQ total mean: {df['eq_total'].mean():.2f}")

# STEP 3: SQR-10 Scoring (Binary 0-1 with reverse-scoring)
print(f"\nSTEP 3: SQR-10 Scoring (Binary 0-1)...")

if sqr_cols:
    # SQR-10 scoring: Binary 0-1 per item
    # Items 1,3,5,7,9: Agree responses (3,4) = 1 point
    # Items 2,4,6,8,10: Disagree responses (1,2) = 1 point (reverse-scored)
    
    sqr_reverse_items = [2, 4, 6, 8, 10]  # Items that need reverse scoring
    
    for i, col in enumerate(sqr_cols, 1):
        if col in df.columns:
            if i in sqr_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate SQR total
    df['sqr_total'] = df[sqr_cols].sum(axis=1)
    print(f"  SQR total range: {df['sqr_total'].min()} to {df['sqr_total'].max()}")
    print(f"  SQR total mean: {df['sqr_total'].mean():.2f}")

# STEP 4: AQ-10 Scoring (Binary 0-1 with ARC reverse-scoring)
print(f"\nSTEP 4: AQ-10 Scoring (Binary 0-1 with ARC correction)...")

if aq_cols:
    # AQ-10 scoring: Binary 0-1 per item with ARC correction
    # Items 1,7,8,10: Agree responses (3,4) = 1 point (autistic traits)
    # Items 2,3,4,5,6,9: Disagree responses (1,2) = 1 point (autistic traits, reverse-scored)
    
    aq_reverse_items = [2, 3, 4, 5, 6, 9]  # Items that need reverse scoring
    
    for i, col in enumerate(aq_cols, 1):
        if col in df.columns:
            if i in aq_reverse_items:
                # Reverse scoring: disagree = 1, agree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [1, 2] else 0 if x in [3, 4] else np.nan)
                print(f"  {col} (reverse): {df[col].dropna().head(3).tolist()}")
            else:
                # Normal scoring: agree = 1, disagree = 0
                df[col] = df[col].apply(lambda x: 1 if x in [3, 4] else 0 if x in [1, 2] else np.nan)
                print(f"  {col} (normal): {df[col].dropna().head(3).tolist()}")
    
    # Calculate AQ total
    df['aq_total'] = df[aq_cols].sum(axis=1)
    print(f"  AQ total range: {df['aq_total'].min()} to {df['aq_total'].max()}")
    print(f"  AQ total mean: {df['aq_total'].mean():.2f}")

# STEP 5: Validation and Summary
print(f"\nSTEP 5: Validation and Summary...")

# Check for any remaining non-numeric values
print(f"\nChecking for data quality issues:")
for col in all_questionnaire_cols:
    if col in df.columns:
        non_numeric = df[col].apply(lambda x: not pd.api.types.is_numeric_dtype(type(x)) if pd.notna(x) else False).sum()
        if non_numeric > 0:
            print(f"  WARNING: {col} has {non_numeric} non-numeric values")

# Show questionnaire totals summary
print(f"\nQuestionnaire totals summary:")
if 'eq_total' in df.columns:
    print(f"  EQ total: {df['eq_total'].min()}-{df['eq_total'].max()} (mean: {df['eq_total'].mean():.2f})")
if 'sqr_total' in df.columns:
    print(f"  SQR total: {df['sqr_total'].min()}-{df['sqr_total'].max()} (mean: {df['sqr_total'].mean():.2f})")
if 'aq_total' in df.columns:
    print(f"  AQ total: {df['aq_total'].min()}-{df['aq_total'].max()} (mean: {df['aq_total'].mean():.2f})")

# Check missing data in totals
print(f"\nMissing data in totals:")
if 'eq_total' in df.columns:
    print(f"  EQ total missing: {df['eq_total'].isnull().sum()}")
if 'sqr_total' in df.columns:
    print(f"  SQR total missing: {df['sqr_total'].isnull().sum()}")
if 'aq_total' in df.columns:
    print(f"  AQ total missing: {df['aq_total'].isnull().sum()}")

print(f"\nStep C complete. Dataset shape: {df.shape}")
print("✅ All questionnaires scored (binary 0-1)")
print("✅ Reverse-scoring applied correctly")
print("✅ Total scores calculated")
print("✅ Data ready for target variable creation")

### D. Target variable creation

In [ ]:
print("="*80)
print("STEP D: CRITICAL AQ SCORING CORRECTION")
print("="*80)

print("Applying OFFICIAL AQ-10 scoring rules from Allison, Auyeung & Baron-Cohen (2012)...")
print("Based on the official scoring key provided:")

# Check what AQ columns we have
aq_cols = [col for col in df.columns if col.startswith('aq_')]
print(f"Found AQ columns: {aq_cols}")

# OFFICIAL AQ-10 scoring rules (CORRECTED BASED ON DEBUG ANALYSIS):
# Items 1, 7, 8, 10: "Agree" responses (3,4) indicate autistic trait = 1 point each
# Items 2, 3, 4, 5, 6, 9: "Disagree" responses (1,2) indicate autistic trait = 1 point each
# NOTE: This is the REVERSE of what was previously implemented

agree_items = [1, 7, 8, 10]  # Items where "agree" (3,4) indicates autistic trait
disagree_items = [2, 3, 4, 5, 6, 9]  # Items where "disagree" (1,2) indicates autistic trait

print("\nCORRECTED AQ-10 scoring rules (based on debug analysis):")
print("Items 1, 7, 8, 10: Agree responses (3,4) = 1 point each")
print("Items 2, 3, 4, 5, 6, 9: Disagree responses (1,2) = 1 point each")

aq_scores = np.zeros(len(df))

# Score agree items (responses 3,4 = agree = 1 point)
for item_num in agree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        agree_responses = ((df[col_name] == 3) | (df[col_name] == 4)).astype(int)
        aq_scores += agree_responses
        print(f"Scored {col_name}: {agree_responses.sum()} agree responses (3,4)")
    else:
        print(f"Missing column: {col_name}")

# Score disagree items (responses 1,2 = disagree = 1 point)
for item_num in disagree_items:
    col_name = f'aq_{item_num}'
    if col_name in df.columns:
        disagree_responses = ((df[col_name] == 1) | (df[col_name] == 2)).astype(int)
        aq_scores += disagree_responses
        print(f"Scored {col_name}: {disagree_responses.sum()} disagree responses (1,2)")
    else:
        print(f"Missing column: {col_name}")

# Update AQ total with correct scoring
df['aq_total'] = aq_scores

print(f"\nAQ total range (corrected): {df['aq_total'].min()} to {df['aq_total'].max()}")
print(f"Cases with AQ >= 6: {len(df[df['aq_total'] >= 6])}")
print(f"Cases with AQ < 6: {len(df[df['aq_total'] < 6])}")

# Show AQ distribution by autism status
print("\nAQ distribution by autism status:")
aq_by_autism = df.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std'])
print(aq_by_autism)

# Verify AQ scoring is correct
print(f"\nAQ-10 scoring verification:")
print(f"   Score range: {df['aq_total'].min()}-{df['aq_total'].max()} (should be 0-10)")
print(f"   Clinical threshold (AQ≥6): {len(df[df['aq_total'] >= 6])} cases")
print(f"   Autism cases with AQ≥6: {len(df[(df['autism_target']==1) & (df['aq_total']>=6)])}")

# Check if autism cases now have higher AQ scores (as expected)
autism_mean_aq = df[df['autism_target']==1]['aq_total'].mean()
non_autism_mean_aq = df[df['autism_target']==0]['aq_total'].mean()
print(f"\nClinical validation:")
print(f"   Autism cases mean AQ: {autism_mean_aq:.2f}")
print(f"   Non-autism cases mean AQ: {non_autism_mean_aq:.2f}")
print(f"   Expected: Autism cases should have HIGHER AQ scores")
print(f"   Result: {'✅ CORRECT' if autism_mean_aq > non_autism_mean_aq else '❌ STILL WRONG'}")

print(f"\nStep D complete. Dataset shape: {df.shape}")

**deeeeebug** AQ scoring

In [ ]:
print("="*80)
print("DEBUG: AQ SCORING VERIFICATION")
print("="*80)

# Let's examine the raw AQ responses to understand the scoring
print("Examining raw AQ responses for autism vs non-autism cases...")

# Get a sample of autism and non-autism cases
autism_sample = df[df['autism_target'] == 1].head(5)
non_autism_sample = df[df['autism_target'] == 0].head(5)

print("\nSAMPLE AUTISM CASES (first 5):")
print("AQ responses:")
aq_cols = [f'aq_{i}' for i in range(1, 11)]
for idx, row in autism_sample.iterrows():
    print(f"  Case {idx}: {[row[col] for col in aq_cols]} -> AQ Total: {row['aq_total']}")

print("\nSAMPLE NON-AUTISM CASES (first 5):")
print("AQ responses:")
for idx, row in non_autism_sample.iterrows():
    print(f"  Case {idx}: {[row[col] for col in aq_cols]} -> AQ Total: {row['aq_total']}")

# Check if AQ scoring direction is correct
print("\nAQ SCORING DIRECTION CHECK:")
print("If autism cases should have HIGHER AQ scores, but we're seeing LOWER scores,")
print("then either:")
print("1. AQ scoring logic is backwards, OR")
print("2. Target variable creation is wrong, OR") 
print("3. Data quality issues")

# Let's check the AQ item distributions
print("\nAQ ITEM DISTRIBUTIONS BY AUTISM STATUS:")
for i in range(1, 11):
    col = f'aq_{i}'
    if col in df.columns:
        autism_responses = df[df['autism_target'] == 1][col].value_counts().sort_index()
        non_autism_responses = df[df['autism_target'] == 0][col].value_counts().sort_index()
        print(f"\n{col}:")
        print(f"  Autism: {autism_responses.to_dict()}")
        print(f"  Non-autism: {non_autism_responses.to_dict()}")

print("\n" + "="*80)
print("AQ DEBUGGING COMPLETE")
print("="*80)

### E. Feature Engineering (YBT Adapted - No SPQ Features)


In [ ]:
print("="*80)
print("STEP E: YBT FEATURE ENGINEERING (NO SPQ FEATURES)")
print("="*80)

print("Creating engineered features adapted for YBT (excluding SPQ-related features)...")

# Age group bins
print("\nCreating age group bins...")
if 'age' in df.columns:
    df['age_group'] = pd.cut(df['age'], bins=[0, 18, 30, 45, 60, 100], 
                            labels=['0-18', '19-30', '31-45', '46-60', '61+'])
    age_groups = df['age_group'].value_counts()
    print(f"Age groups created: {age_groups.to_dict()}")
else:
    print("Age column not found, skipping age groups")

# Non-linear transformations
print("\nCreating non-linear transformations...")
if 'aq_total' in df.columns:
    df['log_aq_total'] = np.log1p(df['aq_total'])  # log(1+x) to handle zeros
    print("Created log_aq_total")

if 'age' in df.columns:
    df['sqrt_age'] = np.sqrt(df['age'])
    print("Created sqrt_age")

# Interaction terms (NO SPQ interactions)
print("\nCreating interaction terms (excluding SPQ)...")
if 'aq_total' in df.columns and 'eq_total' in df.columns:
    df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']
    print("Created aq_eq_interaction")

if 'age' in df.columns and 'eq_total' in df.columns:
    df['age_x_eq'] = df['age'] * df['eq_total']
    print("Created age_x_eq")

# Questionnaire ratios (NO SPQ ratios)
print("\nCreating questionnaire ratios (excluding SPQ)...")
if 'eq_total' in df.columns and 'sqr_total' in df.columns:
    df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1)  # +1 to avoid division by zero
    print("Created eq_sqr_ratio")

# High AQ threshold (using corrected AQ scoring)
print("\nCreating high AQ threshold...")
if 'aq_total' in df.columns:
    df['high_aq'] = (df['aq_total'] >= 6).astype(int)
    print(f"High AQ cases (>=6): {df['high_aq'].sum()}")
else:
    df['high_aq'] = 0
    print("AQ total not found, created dummy high_aq")

# STEM occupation detection (if available) - using YBT column names
print("\nCreating STEM occupation feature...")
stem_occupation_codes = {2, 3, 5, 21}
def is_stem(occupation_code):
    try:
        return int(float(occupation_code) in stem_occupation_codes)
    except:
        return 0

# Check for occupation-related columns in YBT
occupation_cols = [col for col in df.columns if 'occupation' in col.lower() or 'edu' in col.lower()]
print(f"Available occupation/education columns: {occupation_cols}")

if 'edu' in df.columns:
    df['is_stem_occupation'] = df['edu'].apply(is_stem)
    print(f"STEM occupation cases: {df['is_stem_occupation'].sum()}")
else:
    df['is_stem_occupation'] = 0
    print("Education/occupation column not found, created dummy is_stem_occupation")

# CORRECTED Sex mapping to numeric
print("\nCreating sex numeric mapping...")
if 'sex' in df.columns:
    # Clean sex column first - remove metadata and convert to numeric
    sex_clean = df['sex'].copy()
    # Remove metadata rows (containing question text)
    mask_sex = ~sex_clean.astype(str).str.contains('sex|question', case=False, na=False)
    sex_clean = sex_clean.where(mask_sex, np.nan)
    
    # Convert text responses to numeric codes
    sex_map = {'Male': 0, 'Female': 1, 'Other': 2, 'Prefer not to say': 3}
    df['sex_num'] = sex_clean.map(sex_map)
    
    # Fill any remaining missing values with mode
    if df['sex_num'].isnull().sum() > 0:
        mode_val = df['sex_num'].mode().iloc[0] if len(df['sex_num'].mode()) > 0 else 0
        df['sex_num'] = df['sex_num'].fillna(mode_val)
    
    print(f"Sex distribution: {df['sex_num'].value_counts().to_dict()}")
    print("Sex mapping successful - no zero variance issue")
else:
    # If sex column doesn't exist, this indicates an error in missing value handling
    print("ERROR: Sex column missing - check missing value handling")
    print("Creating fallback sex_num column...")
    df['sex_num'] = 0  # Default fallback
    print("Fallback sex mapping created")

# Additional interactions (NO SPQ interactions)
print("\nCreating additional interactions (excluding SPQ)...")
if 'age' in df.columns and 'aq_total' in df.columns:
    df['age_x_aq'] = df['age'] * df['aq_total']
    print("Created age_x_aq")

if 'sex_num' in df.columns and 'eq_total' in df.columns:
    df['sex_x_eq'] = df['sex_num'] * df['eq_total']
    print("Created sex_x_eq")

# Handle columns that may have been dropped during standardization (using YBT column names)
if 'hand' in df.columns and 'aq_total' in df.columns:
    # Clean hand column first - remove metadata and convert to numeric
    hand_clean = df['hand'].copy()
    # Remove metadata rows (containing question text)
    mask_hand = ~hand_clean.astype(str).str.contains('handedness|question', case=False, na=False)
    hand_clean = hand_clean.where(mask_hand, np.nan)
    
    # Convert to numeric codes
    hand_clean = pd.Categorical(hand_clean).codes
    df['handedness_x_aq'] = hand_clean * df['aq_total']
    print("Created handedness_x_aq")
else:
    df['handedness_x_aq'] = 0  # Create dummy column
    print("hand column not found or AQ not available, created dummy handedness_x_aq")

if 'edu' in df.columns and 'aq_total' in df.columns:
    # Clean edu column first - remove metadata and convert to numeric
    edu_clean = df['edu'].copy()
    # Remove metadata rows (containing question text)
    mask_edu = ~edu_clean.astype(str).str.contains('education|question', case=False, na=False)
    edu_clean = edu_clean.where(mask_edu, np.nan)
    
    # Convert to numeric codes
    edu_clean = pd.Categorical(edu_clean).codes
    df['education_x_aq'] = edu_clean * df['aq_total']
    print("Created education_x_aq")
else:
    df['education_x_aq'] = 0  # Create dummy column
    print("edu column not found or AQ not available, created dummy education_x_aq")

print("Created age_x_aq, sex_x_eq, handedness_x_aq, education_x_aq")

print(f"\nStep E complete. Dataset shape: {df.shape}")
print(f"Total features created: {len(df.columns)}")


### F. Data Standardization and Encoding (YBT Adapted)


In [ ]:
print("="*80)
print("STEP F: YBT DATA STANDARDIZATION AND ENCODING")
print("="*80)

# Apply StandardScaler to questionnaire items (but preserve raw totals for filtering)
print("Standardizing questionnaire items...")
questionnaire_cols = [col for col in df.columns if col.startswith(('eq10_', 'sq10_', 'aq_'))]
# Exclude totals from standardization - we need raw scores for filtering
individual_item_cols = [col for col in questionnaire_cols if not col.endswith('_total')]
scaler = StandardScaler()
if individual_item_cols:
    df[individual_item_cols] = scaler.fit_transform(df[individual_item_cols])
    print(f"Standardized {len(individual_item_cols)} individual questionnaire items")
    print("Preserved raw totals (eq_total, sqr_total, aq_total) for filtering")
else:
    print("No individual questionnaire items found for standardization")

# One-hot encode age groups (sex will be handled separately)
print("\nOne-hot encoding age groups...")
if 'age_group' in df.columns:
    df = pd.get_dummies(df, columns=['age_group'], drop_first=True)
    print("Age groups one-hot encoded")
else:
    print("Age group column not found, skipping one-hot encoding")

# Remove data leakage columns (all diagnosis-related columns)
print("\nRemoving data leakage columns...")
diagnosis_cols = [col for col in df.columns if col.startswith('diagnosis')]
df = df.drop(columns=diagnosis_cols, errors='ignore')
print(f"Removed {len(diagnosis_cols)} diagnosis columns")

# Drop unnecessary columns (YBT-specific)
print("\nDropping unnecessary columns...")
drop_cols = ['Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 
             'ethn', 'English', 'diagnosis_69_TEXT', 'Q311', 'Q56', 'Q57', 'Q58', 'Q59', 'Q60',
             '18. how many childr ', '19. non biological ', 'Q64', 'Q65', 'Q549', 'Q573', 'Q573_30_TEXT',
             'Q314', 'Q314_30_TEXT', 'Q315', 'Q315_1_TEXT', 'Q318', 'Q318_1_TEXT', 'Q317', 'Q317_1_TEXT',
             'Q319', 'Q319_1_TEXT', 'Q320', 'Q311.1', 'Q311_51_TEXT', 'Q329', 'Q330', 'Q321', 'Q322', 
             'Q323', 'Q324', 'Q326', 'Q327', 'SC7', 'SC8', 'SC52']
drop_cols += [col for col in df.columns if col.startswith('Q')]  # Drop all Q columns
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
print(f"Dropped unnecessary columns")

# Convert all remaining categorical columns to numeric
print("\nConverting remaining categorical columns to numeric...")
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Found categorical columns: {list(categorical_cols)}")

for col in categorical_cols:
    if col != 'autism_target':  # Don't convert target variable
        print(f"  Converting {col} to numeric codes...")
        df[col] = pd.Categorical(df[col]).codes
        print(f"    {col} converted to numeric")

# Fill remaining NaNs with 0
print("\nFilling remaining NaNs with 0...")
df = df.fillna(0)

# Save processed data
print("\nSaving processed data...")
output_path = 'data/processed/ybt_processed.csv'
import os
os.makedirs('data/processed', exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Processed data saved to {output_path}. Shape: {df.shape}")

print(f"\nStep F complete. Dataset shape: {df.shape}")
print(f"Final feature count: {len(df.columns)}")


In [ ]:
print("="*80)
print("STEP G.5: AQ-BASED FILTERING (BEFORE DATA LEAKAGE PREVENTION)")
print("="*80)

# Apply AQ-based filtering BEFORE removing AQ features
print("Applying AQ-based filtering: EXCLUDE autism cases with AQ < 6...")
print("This ensures only clinically meaningful autism cases are retained.")

# Check AQ distribution before filtering
if 'aq_total' in df.columns:
    print(f"\nAQ distribution before filtering:")
    aq_dist = df['aq_total'].value_counts().sort_index()
    print(aq_dist)
    
    # Show AQ distribution by autism status
    print("\nAQ distribution by autism status:")
    aq_by_autism = df.groupby('autism_target')['aq_total'].agg(['count', 'mean', 'std'])
    print(aq_by_autism)
    
    # Filter autism cases with AQ >= 6 (EXCLUDE those with AQ < 6)
    autism_cases = df[df['autism_target'] == 1]
    autism_high_aq = autism_cases[autism_cases['aq_total'] >= 6]  # Keep only high AQ autism cases
    autism_low_aq = autism_cases[autism_cases['aq_total'] < 6]    # These will be excluded
    
    print(f"\nAutism cases before filtering: {len(autism_cases)}")
    print(f"Autism cases with AQ < 6 (EXCLUDED): {len(autism_low_aq)}")
    print(f"Autism cases with AQ >= 6 (KEPT): {len(autism_high_aq)}")
    
    # Validate filtering worked correctly
    if len(autism_low_aq) > 0:
        print(f"⚠️  WARNING: {len(autism_low_aq)} autism cases with AQ < 6 will be excluded")
        print("This is expected behavior - we want only clinically meaningful autism cases")
    else:
        print("✅ All autism cases have AQ >= 6 (clinically meaningful)")
    
    # Create filtered dataset - KEEP only autism cases with AQ >= 6
    non_autism_cases = df[df['autism_target'] == 0]
    df_filtered = pd.concat([autism_high_aq, non_autism_cases], ignore_index=True)
    
    # Shuffle filtered dataset
    df_filtered = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\nFiltered dataset:")
    print(f"Shape: {df_filtered.shape}")
    print(f"Target distribution:")
    filtered_counts = df_filtered['autism_target'].value_counts()
    print(filtered_counts)
    print(f"Filtered autism percentage: {df_filtered['autism_target'].mean()*100:.2f}%")
    
    # Post-filtering rebalancing
    print("\nPost-filtering rebalancing...")
    target_counts = df_filtered['autism_target'].value_counts()
    print(f"Current distribution: {target_counts.to_dict()}")
    
    if len(target_counts) == 2 and abs(target_counts[0] - target_counts[1]) > 0:
        # Rebalance if needed
        autism_cases = df_filtered[df_filtered['autism_target'] == 1]
        non_autism_cases = df_filtered[df_filtered['autism_target'] == 0]
        
        min_size = min(len(autism_cases), len(non_autism_cases))
        print(f"Rebalancing to {min_size} cases per group...")
        
        autism_sample = autism_cases.sample(n=min_size, random_state=42)
        non_autism_sample = non_autism_cases.sample(n=min_size, random_state=42)
        
        df_final = pd.concat([autism_sample, non_autism_sample], ignore_index=True)
        df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
        
        df_filtered = df_final.copy()
        
        print(f"Final balanced dataset:")
        print(f"Shape: {df_filtered.shape}")
        final_counts = df_filtered['autism_target'].value_counts()
        print(f"Final distribution: {final_counts.to_dict()}")
    
    # Update df to use filtered dataset
    df = df_filtered.copy()
    
    print(f"\nStep G.5 complete. Dataset shape: {df.shape}")
    print("✅ AQ-based filtering applied successfully")
    print("✅ Only autism cases with AQ >= 6 retained")
    
else:
    print("❌ ERROR: AQ total column not found!")
    print("Cannot apply AQ-based filtering without AQ scores")
    print("Check that AQ scoring was completed in Step D")


# F.5 Data leakage prevention

In [ ]:
print("="*80)
print("STEP F.5: COMPLETE DATA LEAKAGE PREVENTION")
print("="*80)

# Remove ALL AQ-related features to prevent data leakage
print("Removing ALL AQ-related features to prevent data leakage...")

aq_features_to_remove = []
for col in df.columns:
    if 'aq' in col.lower():
        aq_features_to_remove.append(col)

print(f"AQ features to remove: {aq_features_to_remove}")

# Remove AQ features
df = df.drop(columns=aq_features_to_remove, errors='ignore')
print(f"Removed {len(aq_features_to_remove)} AQ-related features")

# Verify no AQ features remain
remaining_aq_features = [col for col in df.columns if 'aq' in col.lower()]
print(f"Remaining AQ features: {remaining_aq_features}")

if len(remaining_aq_features) > 0:
    print("⚠️  WARNING: AQ features still present!")
else:
    print("✅ All AQ features successfully removed")

# Save cleaned dataset
print("\nSaving AQ-free dataset...")
output_path = 'data/processed/ybt_processed_no_aq.csv'
df.to_csv(output_path, index=False)
print(f"AQ-free dataset saved to {output_path}. Shape: {df.shape}")

print(f"\nStep F.5 complete. Dataset shape: {df.shape}")
print(f"Features remaining: {len(df.columns)}")

### G. Data Balancing (50/50 Split)


In [ ]:
print("="*80)
print("STEP G: YBT DATA BALANCING (50/50 SPLIT)")
print("="*80)

# Check current target distribution
print("Current target distribution:")
target_counts = df['autism_target'].value_counts()
print(target_counts)
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# Create balanced dataset (50/50 split)
print("\nCreating balanced dataset...")

# Separate autism and non-autism cases
autism_cases = df[df['autism_target'] == 1]
non_autism_cases = df[df['autism_target'] == 0]

print(f"Autism cases: {len(autism_cases)}")
print(f"Non-autism cases: {len(non_autism_cases)}")

# Determine sample size (use smaller group size)
min_size = min(len(autism_cases), len(non_autism_cases))
print(f"Using sample size: {min_size} per group")

# Sample equal numbers from each group
np.random.seed(42)
autism_sample = autism_cases.sample(n=min_size, random_state=42)
non_autism_sample = non_autism_cases.sample(n=min_size, random_state=42)

# Combine balanced samples
df_balanced = pd.concat([autism_sample, non_autism_sample], ignore_index=True)

# Shuffle the balanced dataset
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset created:")
print(f"Shape: {df_balanced.shape}")
print(f"Target distribution:")
balanced_counts = df_balanced['autism_target'].value_counts()
print(balanced_counts)
print(f"Balanced autism percentage: {df_balanced['autism_target'].mean()*100:.2f}%")

# Update df to use balanced dataset
df = df_balanced.copy()

print(f"\nStep G complete. Dataset shape: {df.shape}")


### H. Final Dataset Filtering (Exclude Autism Cases with AQ < 6)


In [ ]:
print("="*80)
print("STEP H: YBT FINAL DATASET FILTERING")
print("="*80)

print("AQ-based filtering has been moved to Step G.5 (before data leakage prevention)")
print("This step is now redundant as filtering was already applied.")
print(f"Current dataset shape: {df.shape}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")

print(f"\nStep H complete. Dataset shape: {df.shape}")

### I. Data Cleaning (Duplicate Removal, Zero Variance Features)


In [ ]:
print("="*80)
print("STEP I: YBT DATA CLEANING")
print("="*80)

# Remove duplicates
print("Removing duplicates...")
initial_shape = df.shape
df = df.drop_duplicates()
final_shape = df.shape
duplicates_removed = initial_shape[0] - final_shape[0]
print(f"Removed {duplicates_removed} duplicate rows")
print(f"Shape after duplicate removal: {df.shape}")

# Remove zero variance features
print("\nRemoving zero variance features...")
initial_features = len(df.columns)

# Separate target from features
target_col = 'autism_target'
feature_cols = [col for col in df.columns if col != target_col]

# Check for zero variance features (handle both numeric and categorical columns)
zero_var_features = []
for col in feature_cols:
    try:
        # For numeric columns, check variance
        if pd.api.types.is_numeric_dtype(df[col]):
            if df[col].var() == 0:
                zero_var_features.append(col)
        else:
            # For categorical columns, check if all values are the same
            if df[col].nunique() <= 1:
                zero_var_features.append(col)
    except (TypeError, ValueError):
        # If there's an error (e.g., mixed data types), skip this column
        print(f"  Skipping column {col} due to data type issues")
        continue

print(f"Found {len(zero_var_features)} zero variance features: {zero_var_features}")

# Remove zero variance features
if zero_var_features:
    df = df.drop(columns=zero_var_features)
    print(f"Removed {len(zero_var_features)} zero variance features")

final_features = len(df.columns)
print(f"Features before cleaning: {initial_features}")
print(f"Features after cleaning: {final_features}")
print(f"Features removed: {initial_features - final_features}")

# Final dataset summary
print(f"\nFinal cleaned dataset:")
print(f"Shape: {df.shape}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")

# Save final cleaned dataset
print("\nSaving final cleaned dataset...")
final_output_path = 'data/processed/ybt_final_recreated_cleaned.csv'
df.to_csv(final_output_path, index=False)
print(f"Final dataset saved to {final_output_path}")

print(f"\nStep I complete. Final dataset shape: {df.shape}")
print(f"Total features: {len(df.columns)}")
print(f"Total samples: {len(df)}")


## 2. YBT EXPERIMENTAL SETUPS

### A. Baseline Models: Predicting Autism Target (Without AQ Items)


In [ ]:
print("="*80)
print("EXPERIMENT A: YBT BASELINE MODELS - COMPLETE AQ EXCLUSION")
print("="*80)

print(f"Dataset shape: {df.shape}")

# Prepare features and target
print("\nPreparing features and target...")

# AQ features should already be removed in Step F.5
# Verify no AQ features remain
aq_features_remaining = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features remaining: {aq_features_remaining}")

if len(aq_features_remaining) > 0:
    print("⚠️  ERROR: AQ features still present! Remove them first.")
    df = df.drop(columns=aq_features_remaining, errors='ignore')
    print(f"Removed {len(aq_features_remaining)} remaining AQ features")
else:
    print("✅ No AQ features present - data leakage prevented")

# Create feature matrix excluding AQ-related features
feature_cols = [col for col in df.columns if col != 'autism_target']
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df['autism_target']

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Handle data types and missing values
print("\nHandling data types and missing values...")
print(f"Missing values before: {X.isnull().sum().sum()}")

# Convert categorical columns to numeric
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(categorical_cols)}")

for col in categorical_cols:
    X[col] = pd.Categorical(X[col]).codes
    print(f"  Converted {col} to numeric codes")

# Fill any remaining missing values
X = X.fillna(0)
print(f"Missing values after: {X.isnull().sum().sum()}")

# Check for data leakage
print("\nChecking for data leakage...")
feature_correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 feature correlations with target:")
for i, (feature, corr) in enumerate(feature_correlations.head(10).items()):
    print(f"  {feature}: {corr:.4f}")

# Flag high correlations
high_corr_features = feature_correlations[feature_correlations > 0.7]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with high correlation (>0.7):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No high correlation features (correlation < 0.7)")

# Investigate EQ data leakage specifically
print("\n🔍 INVESTIGATING EQ DATA LEAKAGE:")
eq_features = [col for col in X.columns if col.startswith('eq')]
if eq_features:
    eq_correlations = X[eq_features].corrwith(y).abs().sort_values(ascending=False)
    print(f"EQ feature correlations with autism target:")
    for feature, corr in eq_correlations.items():
        print(f"  {feature}: {corr:.4f}")
    
    # Check if EQ correlations are suspiciously high
    high_eq_corr = eq_correlations[eq_correlations > 0.2]  # Lowered threshold to 0.2 for investigation
    if len(high_eq_corr) > 0:
        print(f"\n⚠️  INVESTIGATING: {len(high_eq_corr)} EQ features with correlation > 0.2:")
        for feature, corr in high_eq_corr.items():
            print(f"  {feature}: {corr:.4f}")
        
        print("\n🔍 DETAILED EQ LEAKAGE ANALYSIS:")
        print("Checking if EQ features contain autism-related information...")
        
        # Load original data to check EQ responses by autism status
        try:
            df_original = pd.read_csv('data/processed/ybt_processed.csv')
            if 'autism_target' in df_original.columns:
                # Check EQ responses for autism vs non-autism cases
                autism_cases = df_original[df_original['autism_target'] == 1]
                non_autism_cases = df_original[df_original['autism_target'] == 0]
                
                print(f"\nEQ Response Analysis (Original Dataset):")
                print(f"Autism cases: {len(autism_cases)}")
                print(f"Non-autism cases: {len(non_autism_cases)}")
                
                # Check EQ total scores
                if 'eq_total' in df_original.columns:
                    autism_eq_mean = autism_cases['eq_total'].mean()
                    non_autism_eq_mean = non_autism_cases['eq_total'].mean()
                    print(f"\nEQ Total Scores:")
                    print(f"  Autism cases mean EQ: {autism_eq_mean:.2f}")
                    print(f"  Non-autism cases mean EQ: {non_autism_eq_mean:.2f}")
                    print(f"  Difference: {abs(autism_eq_mean - non_autism_eq_mean):.2f}")
                    
                    # Check if difference is clinically meaningful
                    if abs(autism_eq_mean - non_autism_eq_mean) > 0.5:
                        print(f"  ⚠️  SIGNIFICANT DIFFERENCE: EQ scores differ by >0.5 points")
                        print(f"  This may indicate that EQ features contain autism-related information")
                        print(f"  Consider excluding EQ features from baseline models")
                    else:
                        print(f"  ✅ MINOR DIFFERENCE: EQ scores differ by <0.5 points")
                        print(f"  This suggests EQ features are not strongly related to autism")
                
                # Check individual EQ items
                eq_items = [col for col in df_original.columns if col.startswith('eq10_')]
                if eq_items:
                    print(f"\nIndividual EQ Item Analysis:")
                    for item in eq_items[:5]:  # Check first 5 items
                        autism_item_mean = autism_cases[item].mean()
                        non_autism_item_mean = non_autism_cases[item].mean()
                        diff = abs(autism_item_mean - non_autism_item_mean)
                        print(f"  {item}: Autism={autism_item_mean:.2f}, Non-autism={non_autism_item_mean:.2f}, Diff={diff:.2f}")
                        
                        if diff > 0.3:
                            print(f"    ⚠️  High difference in {item}")
                        else:
                            print(f"    ✅ Normal difference in {item}")
            
        except Exception as e:
            print(f"Could not load original data for EQ analysis: {e}")
        
        print("\n📋 EQ LEAKAGE ASSESSMENT:")
        print("Based on the analysis above:")
        print("1. If EQ correlations are >0.2 AND EQ scores differ significantly between groups")
        print("2. Then EQ features may contain autism-related information (data leakage)")
        print("3. Consider excluding EQ features from baseline models")
        print("4. Or investigate further to understand the relationship")
        
    else:
        print("\n✅ EQ correlations appear normal (< 0.2)")
        print("No evidence of data leakage in EQ features")
else:
    print("No EQ features found in dataset")

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
print(f"Scaled features shape: {X_scaled.shape}")
print(f"Feature means: {X_scaled.mean().mean():.6f}")
print(f"Feature stds: {X_scaled.std().mean():.6f}")

# Split data
print("\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

# Train and evaluate models
print("\n" + "="*60)
print("TRAINING AND EVALUATING BASELINE MODELS (COMPLETE AQ EXCLUSION)")
print("="*60)

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  AUC: {auc:.4f}")

# Summary of results
print("\n" + "="*60)
print("YBT BASELINE MODELS SUMMARY (COMPLETE AQ EXCLUSION)")
print("="*60)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('auc', ascending=False)

print("\nResults ranked by AUC:")
print(results_df.round(4))

# Save results
results_df.to_csv('data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv')
print(f"\nResults saved to: data/processed/ybt_baseline_models_complete_aq_exclusion_results.csv")

print("\n" + "="*80)
print("EXPERIMENT A COMPLETE (COMPLETE AQ EXCLUSION)")
print("="*80)

### B. PCA Analysis (YBT Adapted - No SPQ)


In [ ]:
print("="*80)
print("EXPERIMENT B: YBT PCA ANALYSIS (NO SPQ)")
print("="*80)

# Prepare features for PCA (EQ and SQR items only - no SPQ, no AQ)
print("Preparing features for PCA analysis...")

# Get questionnaire item columns (excluding totals and AQ items)
eq_items = [col for col in df.columns if col.startswith('eq10_')]
sqr_items = [col for col in df.columns if col.startswith('sq10_')]

print(f"EQ items: {len(eq_items)} - {eq_items}")
print(f"SQR items: {len(sqr_items)} - {sqr_items}")

# Combine EQ and SQR items for PCA
pca_features = eq_items + sqr_items
print(f"Total PCA features: {len(pca_features)}")

if len(pca_features) == 0:
    print("ERROR: No questionnaire items found for PCA!")
    print("Available columns:", list(df.columns))
else:
    # Prepare data for PCA
    X_pca = df[pca_features].copy()
    y_pca = df['autism_target']
    
    print(f"PCA dataset shape: {X_pca.shape}")
    print(f"Target distribution: {y_pca.value_counts().to_dict()}")
    
    # Handle missing values
    X_pca = X_pca.fillna(X_pca.median())
    
    # Scale features
    scaler_pca = StandardScaler()
    X_pca_scaled = scaler_pca.fit_transform(X_pca)
    
    # Apply PCA
    print("\nApplying PCA...")
    pca = PCA()
    X_pca_transformed = pca.fit_transform(X_pca_scaled)
    
    # Analyze explained variance
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    print(f"Explained variance by component:")
    for i, (var, cum_var) in enumerate(zip(explained_variance_ratio, cumulative_variance)):
        print(f"  PC{i+1}: {var:.4f} (cumulative: {cum_var:.4f})")
    
    # Find optimal number of components (95% variance)
    n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
    print(f"\nComponents needed for 95% variance: {n_components_95}")
    
    # Apply PCA with optimal components
    pca_optimal = PCA(n_components=n_components_95)
    X_pca_optimal = pca_optimal.fit_transform(X_pca_scaled)
    
    print(f"PCA-reduced features shape: {X_pca_optimal.shape}")
    
    # Train models with PCA features
    print("\nTraining models with PCA features...")
    
    # Split data
    X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
        X_pca_optimal, y_pca, test_size=0.2, random_state=42, stratify=y_pca
    )
    
    # Define models
    models_pca = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
    }
    
    # Train and evaluate models
    results_pca = {}
    
    for name, model in models_pca.items():
        print(f"\nTraining {name} with PCA features...")
        
        # Train model
        model.fit(X_train_pca, y_train_pca)
        
        # Make predictions
        y_pred_pca = model.predict(X_test_pca)
        y_pred_proba_pca = model.predict_proba(X_test_pca)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test_pca, y_pred_pca)
        f1 = f1_score(y_test_pca, y_pred_pca)
        precision = precision_score(y_test_pca, y_pred_pca)
        recall = recall_score(y_test_pca, y_pred_pca)
        auc = roc_auc_score(y_test_pca, y_pred_proba_pca)
        
        # Store results
        results_pca[name] = {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'auc': auc
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1-score: {f1:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  AUC: {auc:.4f}")
    
    # Summary of PCA results
    print("\n" + "="*60)
    print("YBT PCA MODELS SUMMARY")
    print("="*60)
    
    results_pca_df = pd.DataFrame(results_pca).T
    results_pca_df = results_pca_df.sort_values('auc', ascending=False)
    
    print("\nPCA Results ranked by AUC:")
    print(results_pca_df.round(4))
    
    # Save PCA results
    results_pca_df.to_csv('data/processed/ybt_pca_models_results.csv')
    print(f"\nPCA results saved to: data/processed/ybt_pca_models_results.csv")

print("\n" + "="*80)
print("EXPERIMENT B COMPLETE (PCA ANALYSIS)")
print("="*80)


## 3. DESCRIPTIVE ANALYSIS & DATA VALIDATION

Before proceeding to additional experiments, let's conduct comprehensive descriptive analysis to validate our final dataset.


In [ ]:
print("="*80)
print("COMPREHENSIVE DESCRIPTIVE ANALYSIS OF FINAL YBT DATASET")
print("="*80)

# 1. Dataset Overview
print("1. DATASET OVERVIEW")
print("-" * 40)
print(f"Final dataset shape: {df.shape}")
print(f"Total samples: {len(df)}")
print(f"Total features: {len(df.columns)}")
print(f"Target distribution: {df['autism_target'].value_counts().to_dict()}")
print(f"Autism percentage: {df['autism_target'].mean()*100:.2f}%")

# 2. Questionnaire Score Distributions
print("\n2. QUESTIONNAIRE SCORE DISTRIBUTIONS")
print("-" * 40)
questionnaire_totals = ['eq_total', 'sqr_total', 'aq_total', 'd_score']
for col in questionnaire_totals:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"  Range: {df[col].min():.1f} - {df[col].max():.1f}")
        print(f"  Mean: {df[col].mean():.2f} ± {df[col].std():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        
        # By autism status
        autism_scores = df[df['autism_target']==1][col]
        non_autism_scores = df[df['autism_target']==0][col]
        print(f"  Autism cases: {autism_scores.mean():.2f} ± {autism_scores.std():.2f}")
        print(f"  Non-autism cases: {non_autism_scores.mean():.2f} ± {non_autism_scores.std():.2f}")

# 3. Clinical Validation
print("\n3. CLINICAL VALIDATION")
print("-" * 40)

# Load AQ data temporarily for validation (from before data leakage prevention)
print("Loading AQ data for clinical validation...")
try:
    # Try to load the dataset before AQ removal
    df_with_aq = pd.read_csv('data/processed/ybt_processed.csv')
    if 'aq_total' in df_with_aq.columns:
        # Match the current dataset with the AQ data
        df_validation = df.copy()
        
        # Get the indices that correspond to our current filtered dataset
        # We need to map the current dataset back to the original AQ data
        current_indices = df.index
        df_validation['aq_total'] = df_with_aq.loc[current_indices, 'aq_total'].values
        
        # AQ clinical threshold analysis
        high_aq_cases = len(df_validation[df_validation['aq_total'] >= 6])
        autism_high_aq = len(df_validation[(df_validation['autism_target']==1) & (df_validation['aq_total'] >= 6)])
        autism_low_aq = len(df_validation[(df_validation['autism_target']==1) & (df_validation['aq_total'] < 6)])
        
        print(f"AQ Clinical Threshold (≥6):")
        print(f"  Total high AQ cases: {high_aq_cases} ({high_aq_cases/len(df_validation)*100:.1f}%)")
        print(f"  Autism cases with high AQ: {autism_high_aq}")
        print(f"  Autism cases with low AQ: {autism_low_aq}")
        if autism_high_aq + autism_low_aq > 0:
            print(f"  Autism high AQ rate: {autism_high_aq/(autism_high_aq+autism_low_aq)*100:.1f}%")
        
        # Clinical interpretation
        print(f"\nClinical Interpretation:")
        autism_mean_aq = df_validation[df_validation['autism_target']==1]['aq_total'].mean()
        non_autism_mean_aq = df_validation[df_validation['autism_target']==0]['aq_total'].mean()
        print(f"  Autism cases mean AQ: {autism_mean_aq:.2f}")
        print(f"  Non-autism cases mean AQ: {non_autism_mean_aq:.2f}")
        print(f"  Clinical expectation: Autism cases should have HIGHER AQ scores")
        print(f"  Current finding: {'✅ CORRECT' if autism_mean_aq > non_autism_mean_aq else '❌ COUNTERINTUITIVE'}")
        
        # Validation of filtering
        print(f"\nFiltering Validation:")
        print(f"  Expected: All autism cases should have AQ ≥ 6 (after filtering)")
        print(f"  Actual: {autism_low_aq} autism cases with AQ < 6")
        print(f"  Status: {'✅ FILTERING WORKED' if autism_low_aq == 0 else '❌ FILTERING FAILED'}")
        
        # Additional validation checks
        print(f"\nAdditional Validation Checks:")
        print(f"  AQ score range: {df_validation['aq_total'].min()}-{df_validation['aq_total'].max()} (should be 0-10)")
        print(f"  Total samples: {len(df_validation)}")
        print(f"  Autism samples: {len(df_validation[df_validation['autism_target']==1])}")
        print(f"  Non-autism samples: {len(df_validation[df_validation['autism_target']==0])}")
        
        # Check if the dataset is balanced
        target_balance = df_validation['autism_target'].value_counts()
        if len(target_balance) == 2:
            balance_ratio = target_balance[1] / target_balance[0]
            print(f"  Target balance ratio: {balance_ratio:.2f} (should be close to 1.0 for balanced dataset)")
            print(f"  Balance status: {'✅ BALANCED' if 0.8 <= balance_ratio <= 1.2 else '❌ IMBALANCED'}")
        
    else:
        print("❌ AQ data not available for validation")
        print("Cannot perform clinical validation without AQ scores")
        
except Exception as e:
    print(f"❌ Could not load AQ data for validation: {e}")
    print("This indicates a problem with the data processing pipeline")
    print("Check that AQ scoring was completed and data was saved correctly")

# 4. Feature Correlation Analysis
print("\n4. FEATURE CORRELATION ANALYSIS")
print("-" * 40)
# Get numeric features only
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

# Calculate correlations with target
correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 10 features correlated with autism target:")
for i, (feature, corr) in enumerate(correlations.head(10).items()):
    print(f"  {i+1:2d}. {feature}: {corr:.4f}")

# 5. Data Quality Checks
print("\n5. DATA QUALITY CHECKS")
print("-" * 40)
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Infinite values: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

# Check for constant features
constant_features = []
for col in numeric_features:
    if df[col].nunique() <= 1:
        constant_features.append(col)
print(f"Constant features: {len(constant_features)} - {constant_features}")

print("\n" + "="*80)
print("DESCRIPTIVE ANALYSIS COMPLETE")
print("="*80)


# validation cell

In [ ]:
print("="*80)
print("FINAL VALIDATION: DATA LEAKAGE CHECK")
print("="*80)

# Check for any remaining AQ features
aq_features_check = [col for col in df.columns if 'aq' in col.lower()]
print(f"AQ features in final dataset: {aq_features_check}")

if len(aq_features_check) > 0:
    print("❌ DATA LEAKAGE DETECTED!")
    print("The following AQ features are still present:")
    for feature in aq_features_check:
        print(f"  - {feature}")
    print("\nThis will invalidate the model results!")
else:
    print("✅ No AQ features present - data leakage prevented")

# Check feature correlations with target
print("\nFeature correlations with autism target:")
numeric_features = df.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'autism_target']

correlations = df[numeric_features].corrwith(df['autism_target']).abs().sort_values(ascending=False)
print("Top 5 feature correlations:")
for i, (feature, corr) in enumerate(correlations.head(5).items()):
    print(f"  {i+1}. {feature}: {corr:.4f}")

# Flag any suspiciously high correlations
high_corr_features = correlations[correlations > 0.5]
if len(high_corr_features) > 0:
    print(f"\n⚠️  WARNING: {len(high_corr_features)} features with very high correlation (>0.5):")
    for feature, corr in high_corr_features.items():
        print(f"  {feature}: {corr:.4f}")
else:
    print("\n✅ No suspiciously high correlations detected")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)